# 3.2 — Data Preparation

**Goal:** Telco Customer Churn dataset-কে Machine Learning-এর জন্য প্রস্তুত করা।

এই অংশে data download/read করব, column ও text value uniform করব, `totalcharges` ঠিক করব এবং target `churn`-কে numeric বানাব।

## 1. প্রয়োজনীয় libraries import করা

**কেন?** `pandas` দিয়ে dataset handle করব, `numpy` numerical কাজের জন্য এবং `matplotlib` পরের analysis/visualization-এর কাজে লাগবে।

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Dataset download করা



In [21]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

# !wget $data -O data-week-3.csv

## 3. CSV পড়া ও data দেখা


In [22]:
df = pd.read_csv('data-week-3.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


`df.head()`-এ অনেক column থাকলে মাঝের column-গুলো `...` দিয়ে লুকানো থাকতে পারে। Transpose করলে row ও column-এর অবস্থান বদলে যায়, তাই সব column-এর value একসঙ্গে দেখা সহজ হয়।

In [34]:
df.shape

(7043, 21)

In [23]:
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


## 4. Column name ও text value uniform করা


In [24]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

## 5. Data type পরীক্ষা করা

**কেন?** কোনো column-এর value দেখতে number-এর মতো হলেও pandas সেটিকে string হিসেবে পড়েছে কি না, তা বুঝতে চাই। বিশেষ করে `totalcharges` numeric হওয়া উচিত।

In [25]:
df.dtypes

customerid           object
gender               object
seniorcitizen         int64
partner              object
dependents           object
tenure                int64
phoneservice         object
multiplelines        object
internetservice      object
onlinesecurity       object
onlinebackup         object
deviceprotection     object
techsupport          object
streamingtv          object
streamingmovies      object
contract             object
paperlessbilling     object
paymentmethod        object
monthlycharges      float64
totalcharges         object
churn                object
dtype: object

## 6. `totalcharges`-কে numeric-এ রূপান্তর

Original data-তে কিছু missing value ফাঁকা space দিয়ে লেখা ছিল। আগের step-এ space `_` হয়ে যাওয়ায় `totalcharges`-এ কিছু value `_` হয়েছে; pandas এটিকে number হিসেবে convert করতে পারে না।

In [26]:
df['totalcharges']
# shob to number ei dekha jacche tai na? then numeric e convert kore dekhi tahole ei column ke object boltese ken

0         29.85
1        1889.5
2        108.15
3       1840.75
4        151.65
         ...   
7038     1990.5
7039     7362.9
7040     346.45
7041      306.6
7042     6844.5
Name: totalcharges, Length: 7043, dtype: object

In [27]:
pd.to_numeric(df['totalcharges'])

ValueError: Unable to parse string "_" at position 488

In [ ]:
# so uporer error dekhay: ValueError: Unable to parse string "_" at position 488
# mane 488 index e "_" ache, tai numeric e convert korte parlo na. so ei row ta dekhbo. maybe aaro emon ase. but error er first index tai show kore.

df['totalcharges'].iloc[488]

'_'

In [ ]:
tc = pd.to_numeric(df.totalcharges, errors='coerce')

`errors='coerce'` invalid value-কে `NaN` বানায়, ফলে conversion error দিয়ে থেমে যায় না। এরপর কতগুলো missing value হয়েছে, তা গুনি।

In [ ]:
tc.isnull().sum()

np.int64(11)

## 7. Missing row দেখা ও ঠিক করা

**কেন?** কোন customer-দের `totalcharges` missing, তা পরীক্ষা করে তারপর উপযুক্ত value বসাব।

In [ ]:
df[tc.isnull()][['customerid', 'totalcharges']]

,customerid,totalcharges
488,4472-lvygi,_
753,3115-czmzd,_
936,5709-lvoeq,_
1082,4367-nuyao,_
1340,1371-dwpaz,_
3331,7644-omvmy,_
3826,3213-vvolg,_
4380,2520-sgtta,_
5218,2923-arzlg,_
6670,4075-wkniu,_


এই dataset-এ এমন ১১টি row আছে। এগুলো সাধারণত নতুন customer, যাদের total charge রেকর্ড হয়নি। তাই এখানে missing `totalcharges`-কে `0` দিয়ে পূরণ করছি। অন্য dataset-এ সবসময় `0` উপযুক্ত নাও হতে পারে।

In [ ]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

## 8. Target variable `churn` numeric করা

**কেন?** `churn`-এ `yes`/`no` string আছে। Classification model-এর জন্য এটিকে `1`/`0` করা দরকার।

- `yes` → `1`: customer churn করেছে।
- `no` → `0`: customer churn করেনি।

In [ ]:
df.churn.head()

0     no
1     no
2    yes
3     no
4    yes
Name: churn, dtype: object

In [ ]:
df.churn = (df.churn == 'yes').astype(int)
df.churn.head()

0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64

## Quick recap

1. CSV download করে pandas দিয়ে পড়েছি।
2. Column name ও text value uniform করেছি।
3. `totalcharges` numeric করেছি এবং missing value `0` দিয়ে পূরণ করেছি।
4. `churn`-কে binary target (`1`/`0`) করেছি।

এখন data পরের ধাপ—**train, validation ও test split**—এর জন্য প্রস্তুত।